In [1]:
# ============================================================
# PRETRAINED BERT — INDIAN LEGAL DOMAIN ADAPTATION
# ============================================================
#
# MODEL:
#   bert-base-uncased
#
# DATA:
#   Same InLegal dataset used for GPT-2 and Mamba
#
# SPLIT:
#   90% TRAIN
#   10% VALIDATION
#
# OBJECTIVE:
#   Masked Language Modeling (MLM)
#
# METRIC:
#   Pseudo-Perplexity (PPPL)
#
# IMPORTANT:
#   BERT is NOT a causal language model.
#   Therefore ordinary causal perplexity is not appropriate.
#   We use pseudo-perplexity by masking tokens individually.
#
# ============================================================


# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q -U transformers datasets accelerate huggingface_hub


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import gc
import math
import random
import shutil
import zipfile
import time

import numpy as np
import pandas as pd
import torch

from pathlib import Path

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed
)

from huggingface_hub import hf_hub_download


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

SEED = 61

set_seed(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. ENVIRONMENT
# ============================================================

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print(
        "GPU memory      :",
        round(gpu_memory, 2),
        "GB"
    )


# ============================================================
# 5. MODEL CONFIGURATION
# ============================================================

MODEL_NAME = "bert-base-uncased"

# BERT supports maximum 512 positions

BERT_BLOCK_SIZE = 512

# Your original experiment uses 100,000 x 1024-token blocks.
# Each 1024 block becomes TWO BERT blocks.

MAX_TRAIN_BLOCKS_1024 = 100_000

MAX_VALID_BLOCKS_1024 = 2_000


# ============================================================
# 6. TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 2

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

PER_DEVICE_BATCH_SIZE = 8

GRADIENT_ACCUMULATION_STEPS = 1


# ============================================================
# 7. PPPL EVALUATION CONFIGURATION
# ============================================================

# PPPL is VERY expensive for BERT because every token
# needs to be masked/evaluated.
#
# We therefore use a fixed subset for PPPL evaluation.
#
# The validation split itself remains exactly the same.

PPPL_VALID_BLOCKS = 100

PPPL_MAX_TOKENS_PER_BLOCK = 256


# ============================================================
# 8. OUTPUT
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "bert_legal_finetune"
)

FINAL_MODEL_DIR = (
    "/kaggle/working/"
    "bert_legal_finetune_final"
)

RESULTS_PATH = (
    "/kaggle/working/"
    "bert_legal_finetune_results.csv"
)


# ============================================================
# 9. DATASET DOWNLOAD
# ============================================================

print("\n" + "=" * 70)
print("DOWNLOADING INLEGAL DATASET")
print("=" * 70)


HF_REPO = "L-NLProc/InLegalLlama-training-data"

HF_FILENAME = "For CPT/Train_data.zip"

DATA_DIR = "/kaggle/working/legal_data"

ZIP_PATH = "/kaggle/working/Train_data.zip"


os.makedirs(
    DATA_DIR,
    exist_ok=True
)


try:

    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        repo_type="dataset"
    )

    print(
        "\nDownloaded dataset publicly."
    )


except Exception:

    print(
        "\nPublic download failed."
    )

    print(
        "Trying Kaggle Hugging Face token..."
    )

    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()

    HF_TOKEN = user_secrets.get_secret(
        "HF-MTP-SLM"
    )

    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        repo_type="dataset",
        token=HF_TOKEN
    )

    print(
        "Downloaded using HF token."
    )


shutil.copy(
    downloaded_path,
    ZIP_PATH
)


# ============================================================
# 10. EXTRACT
# ============================================================

print("\n" + "=" * 70)
print("EXTRACTING DATASET")
print("=" * 70)


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zip_ref:

    zip_ref.extractall(
        DATA_DIR
    )


print(
    "Extracted to:",
    DATA_DIR
)


# ============================================================
# 11. FIND CSV
# ============================================================

csv_files = list(
    Path(DATA_DIR).rglob("*.csv")
)


print("\nCSV files found:")

for f in csv_files:

    print(
        " -",
        f
    )


if len(csv_files) == 0:

    raise FileNotFoundError(
        "No CSV found."
    )


CSV_PATH = (
    "/kaggle/working/legal_data/"
    "Train_data/"
    "train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv"
)


if not os.path.exists(
    CSV_PATH
):

    print(
        "\nExpected CSV path not found."
    )

    CSV_PATH = str(
        csv_files[0]
    )


print(
    "\nUsing CSV:"
)

print(
    CSV_PATH
)


# ============================================================
# 12. LOAD DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING RAW LEGAL DATA")
print("=" * 70)


df = pd.read_csv(
    CSV_PATH
)


print(
    "Original shape:",
    df.shape
)

print(
    "Columns:",
    df.columns.tolist()
)


if "text" not in df.columns:

    raise ValueError(
        "'text' column not found."
    )


df = df[
    ["text"]
].copy()


df["text"] = (
    df["text"]
    .astype(str)
)


df = df[
    df["text"].str.strip() != ""
]


# ============================================================
# 13. REMOVE EXACT DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("REMOVING EXACT DUPLICATES")
print("=" * 70)


before_duplicates = len(df)


df = df.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(
    drop=True
)


after_duplicates = len(df)


print(
    "Before:",
    before_duplicates
)

print(
    "Duplicates removed:",
    before_duplicates - after_duplicates
)

print(
    "After:",
    after_duplicates
)


# ============================================================
# 14. 90 / 10 DOCUMENT-LEVEL SPLIT
# ============================================================

print("\n" + "=" * 70)
print("90 / 10 TRAIN / VALIDATION SPLIT")
print("=" * 70)


dataset = Dataset.from_pandas(
    df,
    preserve_index=False
)


split_dataset = (
    dataset.train_test_split(
        test_size=0.10,
        seed=SEED
    )
)


train_dataset = (
    split_dataset["train"]
)

valid_dataset = (
    split_dataset["test"]
)


print(
    "Training documents:",
    len(train_dataset)
)

print(
    "Validation documents:",
    len(valid_dataset)
)


# ============================================================
# 15. LOAD BERT TOKENIZER
# ============================================================

print("\n" + "=" * 70)
print("LOADING BERT TOKENIZER")
print("=" * 70)


tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_NAME
    )
)


print(
    "Tokenizer:",
    MODEL_NAME
)

print(
    "Vocabulary:",
    len(tokenizer)
)

print(
    "CLS token:",
    tokenizer.cls_token
)

print(
    "SEP token:",
    tokenizer.sep_token
)

print(
    "MASK token:",
    tokenizer.mask_token
)

print(
    "PAD token:",
    tokenizer.pad_token
)


# ============================================================
# 16. TOKENIZE DOCUMENTS
# ============================================================

print("\n" + "=" * 70)
print("TOKENIZING LEGAL DOCUMENTS")
print("=" * 70)


def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        add_special_tokens=True,
        truncation=False
    )


tokenized_train = train_dataset.map(

    tokenize_function,

    batched=True,

    num_proc=2,

    remove_columns=["text"],

    desc="Tokenizing training documents"
)


tokenized_valid = valid_dataset.map(

    tokenize_function,

    batched=True,

    num_proc=2,

    remove_columns=["text"],

    desc="Tokenizing validation documents"
)


print(
    "Tokenization complete."
)


# ============================================================
# 17. CREATE 512-TOKEN BERT BLOCKS
# ============================================================

print("\n" + "=" * 70)
print("CREATING 512-TOKEN BERT BLOCKS")
print("=" * 70)


def group_texts(examples):

    concatenated = {}

    for key in examples.keys():

        concatenated[key] = sum(
            examples[key],
            []
        )


    total_length = len(
        concatenated["input_ids"]
    )


    total_length = (
        total_length
        // BERT_BLOCK_SIZE
    ) * BERT_BLOCK_SIZE


    result = {

        key: [

            tokens[
                i:i + BERT_BLOCK_SIZE
            ]

            for i in range(
                0,
                total_length,
                BERT_BLOCK_SIZE
            )
        ]

        for key, tokens
        in concatenated.items()
    }


    return result


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

bert_train_full = (
    tokenized_train.map(

        group_texts,

        batched=True,

        batch_size=1000,

        num_proc=2,

        desc="Creating training blocks"
    )
)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

bert_valid_full = (
    tokenized_valid.map(

        group_texts,

        batched=True,

        batch_size=1000,

        num_proc=2,

        desc="Creating validation blocks"
    )
)


print(
    "\nFull BERT training blocks:",
    len(bert_train_full)
)

print(
    "Full BERT validation blocks:",
    len(bert_valid_full)
)


# ============================================================
# 18. SELECT EXPERIMENT DATA
# ============================================================

print("\n" + "=" * 70)
print("SELECTING EXPERIMENT DATA")
print("=" * 70)


bert_train_full = (
    bert_train_full.shuffle(
        seed=SEED
    )
)


# IMPORTANT:
#
# We use 2 x 512-token blocks for every original
# 1024-token block.
#
# Therefore, use twice as many BERT blocks.

MAX_TRAIN_BERT_BLOCKS = (
    MAX_TRAIN_BLOCKS_1024 * 2
)

MAX_VALID_BERT_BLOCKS = (
    MAX_VALID_BLOCKS_1024 * 2
)


train_blocks_to_use = min(

    MAX_TRAIN_BERT_BLOCKS,

    len(bert_train_full)
)


valid_blocks_to_use = min(

    MAX_VALID_BERT_BLOCKS,

    len(bert_valid_full)
)


bert_train = (
    bert_train_full.select(
        range(
            train_blocks_to_use
        )
    )
)


bert_valid = (
    bert_valid_full.select(
        range(
            valid_blocks_to_use
        )
    )
)


print(
    "Training BERT blocks:",
    len(bert_train)
)

print(
    "Validation BERT blocks:",
    len(bert_valid)
)


print(
    "\nTraining tokens:",
    f"{len(bert_train) * 512:,}"
)

print(
    "Validation tokens:",
    f"{len(bert_valid) * 512:,}"
)


# ============================================================
# 19. DATA SANITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("DATA SANITY CHECK")
print("=" * 70)


sample_ids = (
    bert_train[0]["input_ids"]
)


print(
    "Sample block length:",
    len(sample_ids)
)


print(
    "\nDecoded sample:"
)


print(
    tokenizer.decode(
        sample_ids[:300]
    )
)


# ============================================================
# 20. MLM DATA COLLATOR
# ============================================================

print("\n" + "=" * 70)
print("CREATING MLM COLLATOR")
print("=" * 70)


data_collator = (
    DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=True,

        mlm_probability=0.15
    )
)


print(
    "Objective:",
    "Masked Language Modeling"
)

print(
    "Mask probability:",
    0.15
)


# ============================================================
# 21. LOAD PRETRAINED BERT
# ============================================================

print("\n" + "=" * 70)
print("LOADING PRETRAINED BERT")
print("=" * 70)


model = (
    AutoModelForMaskedLM
    .from_pretrained(
        MODEL_NAME
    )
)


num_parameters = sum(

    p.numel()

    for p in model.parameters()
)


trainable_parameters = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    "Model loaded successfully."
)


print(
    "\nTotal parameters:",
    f"{num_parameters:,}"
)

print(
    "Total parameters (M):",
    f"{num_parameters / 1e6:.2f}M"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ============================================================
# 22. TRAINING ARGUMENTS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)


print(
    "Epochs:",
    NUM_EPOCHS
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Batch size:",
    PER_DEVICE_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)


training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    warmup_steps=0,

    lr_scheduler_type="cosine",

    per_device_train_batch_size=(
        PER_DEVICE_BATCH_SIZE
    ),

    per_device_eval_batch_size=(
        PER_DEVICE_BATCH_SIZE
    ),

    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    logging_strategy="steps",

    logging_steps=100,

    fp16=torch.cuda.is_available(),

    gradient_checkpointing=True,

    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    seed=SEED,

    data_seed=SEED,

    report_to="none",

    remove_unused_columns=True
)


# ============================================================
# 23. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=bert_train,

    eval_dataset=bert_valid,

    data_collator=data_collator,

    processing_class=tokenizer
)


print(
    "\nTrainer created successfully."
)


# ============================================================
# 24. BASELINE MLM VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("BASELINE — BEFORE FINE-TUNING")
print("=" * 70)


baseline_start = time.time()


baseline_results = (
    trainer.evaluate()
)


baseline_time = (
    time.time()
    - baseline_start
)


baseline_mlm_loss = (
    baseline_results["eval_loss"]
)


print(
    "\nBaseline MLM loss:",
    f"{baseline_mlm_loss:.6f}"
)


print(
    "Baseline MLM pseudo-perplexity:",
    f"{math.exp(baseline_mlm_loss):.6f}"
)


print(
    "Baseline evaluation time:",
    f"{baseline_time / 60:.2f} minutes"
)


# ============================================================
# 25. CLEAR CACHE
# ============================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 26. TRAIN
# ============================================================

print("\n" + "=" * 70)
print("STARTING BERT LEGAL DOMAIN ADAPTATION")
print("=" * 70)


training_start = time.time()


train_result = (
    trainer.train()
)


training_time = (
    time.time()
    - training_start
)


print(
    "\nTraining completed."
)

print(
    "Training time:",
    f"{training_time / 3600:.2f} hours"
)


# ============================================================
# 27. FINAL MLM VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL MLM VALIDATION")
print("=" * 70)


final_results = (
    trainer.evaluate()
)


final_mlm_loss = (
    final_results["eval_loss"]
)


print(
    "\nFinal MLM loss:",
    f"{final_mlm_loss:.6f}"
)


print(
    "Final MLM pseudo-perplexity:",
    f"{math.exp(final_mlm_loss):.6f}"
)


# ============================================================
# 28. PSEUDO-PERPLEXITY FUNCTION
# ============================================================
#
# This is the IMPORTANT BERT metric.
#
# For each token:
#
#   original sentence
#          ↓
#   mask one token
#          ↓
#   BERT predicts token
#          ↓
#   obtain probability of original token
#
# PPPL = exp(
#       average negative log probability
# )
#
# ============================================================


def calculate_pseudo_perplexity(
    model,
    tokenizer,
    dataset,
    num_blocks=100,
    max_tokens_per_block=256
):

    model.eval()

    device = next(
        model.parameters()
    ).device

    total_log_probability = 0.0

    total_tokens = 0


    num_blocks = min(
        num_blocks,
        len(dataset)
    )


    print(
        "\nCalculating pseudo-perplexity..."
    )

    print(
        "Validation blocks:",
        num_blocks
    )

    print(
        "Max tokens per block:",
        max_tokens_per_block
    )


    for block_idx in range(
        num_blocks
    ):

        input_ids = torch.tensor(
            dataset[block_idx]["input_ids"],
            dtype=torch.long
        )


        # Avoid CLS and SEP

        start = 1

        end = min(
            len(input_ids) - 1,
            max_tokens_per_block + 1
        )


        input_ids = (
            input_ids[start:end]
        )


        attention_mask = torch.ones(
            len(input_ids),
            dtype=torch.long
        )


        for token_idx in range(
            len(input_ids)
        ):

            masked_input = (
                input_ids.clone()
            )


            original_token = (
                masked_input[token_idx]
                .item()
            )


            # Do not evaluate special tokens

            if original_token in [
                tokenizer.cls_token_id,
                tokenizer.sep_token_id,
                tokenizer.pad_token_id
            ]:

                continue


            masked_input[token_idx] = (
                tokenizer.mask_token_id
            )


            inputs = {

                "input_ids":
                    masked_input.unsqueeze(0).to(device),

                "attention_mask":
                    attention_mask.unsqueeze(0).to(device)
            }


            with torch.no_grad():

                outputs = model(
                    **inputs
                )


            logits = (
                outputs.logits[
                    0,
                    token_idx
                ]
            )


            log_probs = torch.log_softmax(
                logits,
                dim=-1
            )


            token_log_probability = (
                log_probs[original_token]
                .item()
            )


            total_log_probability += (
                token_log_probability
            )

            total_tokens += 1


        if (
            block_idx + 1
        ) % 10 == 0:

            print(
                f"Processed "
                f"{block_idx + 1}/"
                f"{num_blocks} blocks"
            )


    average_negative_log_probability = (
        -total_log_probability
        / total_tokens
    )


    pseudo_perplexity = math.exp(
        average_negative_log_probability
    )


    return (
        pseudo_perplexity,
        total_tokens
    )


# ============================================================
# 29. BASELINE PSEUDO-PERPLEXITY
# ============================================================

print("\n" + "=" * 70)
print("BASELINE PSEUDO-PERPLEXITY")
print("=" * 70)


# Reload original pretrained BERT
# so baseline PPPL is measured BEFORE fine-tuning.

baseline_model = (
    AutoModelForMaskedLM
    .from_pretrained(
        MODEL_NAME
    )
)


baseline_model.to(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


baseline_pppl_start = time.time()


baseline_pppl, pppl_tokens = (
    calculate_pseudo_perplexity(

        baseline_model,

        tokenizer,

        bert_valid,

        num_blocks=PPPL_VALID_BLOCKS,

        max_tokens_per_block=
            PPPL_MAX_TOKENS_PER_BLOCK
    )
)


baseline_pppl_time = (
    time.time()
    - baseline_pppl_start
)


print(
    "\nBaseline BERT pseudo-perplexity:",
    f"{baseline_pppl:.6f}"
)

print(
    "Tokens evaluated:",
    pppl_tokens
)

print(
    "Time:",
    f"{baseline_pppl_time / 60:.2f} minutes"
)


del baseline_model

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 30. FINAL PSEUDO-PERPLEXITY
# ============================================================

print("\n" + "=" * 70)
print("FINAL PSEUDO-PERPLEXITY")
print("=" * 70)


model = trainer.model

model.to(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


final_pppl_start = time.time()


final_pppl, final_pppl_tokens = (
    calculate_pseudo_perplexity(

        model,

        tokenizer,

        bert_valid,

        num_blocks=PPPL_VALID_BLOCKS,

        max_tokens_per_block=
            PPPL_MAX_TOKENS_PER_BLOCK
    )
)


final_pppl_time = (
    time.time()
    - final_pppl_start
)


print(
    "\nFinal BERT pseudo-perplexity:",
    f"{final_pppl:.6f}"
)

print(
    "Tokens evaluated:",
    final_pppl_tokens
)

print(
    "Time:",
    f"{final_pppl_time / 60:.2f} minutes"
)


# ============================================================
# 31. IMPROVEMENT
# ============================================================

pppl_improvement = (
    baseline_pppl
    - final_pppl
)


pppl_improvement_percent = (

    pppl_improvement
    / baseline_pppl
    * 100
)


print("\n" + "=" * 70)
print("BERT DOMAIN ADAPTATION RESULTS")
print("=" * 70)


print(
    "\nBaseline pseudo-perplexity:",
    f"{baseline_pppl:.6f}"
)

print(
    "Final pseudo-perplexity:",
    f"{final_pppl:.6f}"
)

print(
    "PPPL improvement:",
    f"{pppl_improvement:.6f}"
)

print(
    "PPPL improvement (%):",
    f"{pppl_improvement_percent:.2f}%"
)


# ============================================================
# 32. SAVE MODEL
# ============================================================

print("\n" + "=" * 70)
print("SAVING FINE-TUNED BERT")
print("=" * 70)


trainer.save_model(
    FINAL_MODEL_DIR
)


tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)


print(
    "\nModel saved to:"
)

print(
    FINAL_MODEL_DIR
)


# ============================================================
# 33. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame({

    "model": [
        "BERT-base"
    ],

    "model_name": [
        MODEL_NAME
    ],

    "architecture": [
        "BERT / Transformer Encoder"
    ],

    "initialization": [
        "Pretrained"
    ],

    "objective": [
        "Masked Language Modeling"
    ],

    "train_documents": [
        len(train_dataset)
    ],

    "validation_documents": [
        len(valid_dataset)
    ],

    "original_block_size": [
        1024
    ],

    "bert_block_size": [
        512
    ],

    "train_blocks_used": [
        len(bert_train)
    ],

    "validation_blocks_used": [
        len(bert_valid)
    ],

    "epochs": [
        NUM_EPOCHS
    ],

    "learning_rate": [
        LEARNING_RATE
    ],

    "batch_size": [
        PER_DEVICE_BATCH_SIZE
    ],

    "baseline_mlm_loss": [
        baseline_mlm_loss
    ],

    "final_mlm_loss": [
        final_mlm_loss
    ],

    "baseline_mlm_exp_loss": [
        math.exp(baseline_mlm_loss)
    ],

    "final_mlm_exp_loss": [
        math.exp(final_mlm_loss)
    ],

    "baseline_pseudo_perplexity": [
        baseline_pppl
    ],

    "final_pseudo_perplexity": [
        final_pppl
    ],

    "pppl_improvement": [
        pppl_improvement
    ],

    "pppl_improvement_percent": [
        pppl_improvement_percent
    ],

    "pppl_tokens_evaluated": [
        final_pppl_tokens
    ],

    "training_time_hours": [
        training_time / 3600
    ]
})


results_df.to_csv(
    RESULTS_PATH,
    index=False
)


print(
    "\nResults saved to:"
)

print(
    RESULTS_PATH
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL BERT EXPERIMENT SUMMARY")
print("=" * 70)


print(
    "\nModel:",
    MODEL_NAME
)

print(
    "Train documents:",
    len(train_dataset)
)

print(
    "Validation documents:",
    len(valid_dataset)
)

print(
    "Train BERT blocks:",
    len(bert_train)
)

print(
    "Validation BERT blocks:",
    len(bert_valid)
)


print(
    "\nBaseline MLM loss:",
    f"{baseline_mlm_loss:.6f}"
)

print(
    "Final MLM loss:",
    f"{final_mlm_loss:.6f}"
)


print(
    "\nBaseline pseudo-perplexity:",
    f"{baseline_pppl:.6f}"
)

print(
    "Final pseudo-perplexity:",
    f"{final_pppl:.6f}"
)


print(
    "\nPPPL improvement:",
    f"{pppl_improvement:.6f}"
)

print(
    "PPPL improvement (%):",
    f"{pppl_improvement_percent:.2f}%"
)


print(
    "\nTraining time:",
    f"{training_time / 3600:.2f} hours"
)


print(
    "\nFine-tuned model:"
)

print(
    FINAL_MODEL_DIR
)


print(
    "\nResults CSV:"
)

print(
    RESULTS_PATH
)


print("\n" + "=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ENVIRONMENT
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU me

For CPT/Train_data.zip: reconstructing file:   0%|          |  0.00B /  518MB            

For CPT/Train_data.zip: downloading bytes:           |  0.00B            

Downloaded using HF token.

EXTRACTING DATASET
Extracted to: /kaggle/working/legal_data

CSV files found:
 - /kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv

Using CSV:
/kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv

LOADING RAW LEGAL DATA
Original shape: (138321, 1)
Columns: ['text']

REMOVING EXACT DUPLICATES
Before: 138321
Duplicates removed: 4665
After: 133656

90 / 10 TRAIN / VALIDATION SPLIT
Training documents: 120290
Validation documents: 13366

LOADING BERT TOKENIZER


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Calculating pseudo-perplexity...
Validation blocks: 100
Max tokens per block: 256
Processed 10/100 blocks
Processed 20/100 blocks
Processed 30/100 blocks
Processed 40/100 blocks
Processed 50/100 blocks
Processed 60/100 blocks
Processed 70/100 blocks
Processed 80/100 blocks
Processed 90/100 blocks
Processed 100/100 blocks

Baseline BERT pseudo-perplexity: 10.801816
Tokens evaluated: 25586
Time: 9.56 minutes

FINAL PSEUDO-PERPLEXITY

Calculating pseudo-perplexity...
Validation blocks: 100
Max tokens per block: 256
Processed 10/100 blocks
Processed 20/100 blocks
Processed 30/100 blocks
Processed 40/100 blocks
Processed 50/100 blocks
Processed 60/100 blocks
Processed 70/100 blocks
Processed 80/100 blocks
Processed 90/100 blocks
Processed 100/100 blocks

Final BERT pseudo-perplexity: 3.449331
Tokens evaluated: 25586
Time: 9.55 minutes

BERT DOMAIN ADAPTATION RESULTS

Baseline pseudo-perplexity: 10.801816
Final pseudo-perplexity: 3.449331
PPPL improvement: 7.352485
PPPL improvement (%): 68.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
/kaggle/working/bert_legal_finetune_final

Results saved to:
/kaggle/working/bert_legal_finetune_results.csv

FINAL BERT EXPERIMENT SUMMARY

Model: bert-base-uncased
Train documents: 120290
Validation documents: 13366
Train BERT blocks: 200000
Validation BERT blocks: 4000

Baseline MLM loss: 5.484676
Final MLM loss: 2.732574

Baseline pseudo-perplexity: 10.801816
Final pseudo-perplexity: 3.449331

PPPL improvement: 7.352485
PPPL improvement (%): 68.07%

Training time: 10.52 hours

Fine-tuned model:
/kaggle/working/bert_legal_finetune_final

Results CSV:
/kaggle/working/bert_legal_finetune_results.csv

